In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.cm as cm   
import matplotlib.colors as mcolors
from src import import_data as i_d

/Users/charles.ozeel/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/charles.ozeel/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
i_d.charger_donnees_depuis_bureau()

Dossier trouvé : /Users/charles.ozeel/Desktop/donnees_formule_un
Fichiers CSV trouvés : [PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/circuits.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/status.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/lap_times.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/sprint_results.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/drivers.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/races.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/constructors.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/constructor_standings.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/qualifying.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/driver_standings.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees_formule_un/constructor_results.csv'), PosixPath('/Users/charles.ozeel/Desktop/donnees

In [3]:
def horaire_moyen_run_f1(races: pd.DataFrame):
    
    loca = pd.read_csv("f1_grands_prix_locations.csv").copy(deep=True)
    
    races = races.copy(deep=True)
    
    races.replace("\\N", np.nan, inplace=True)
    
    date = ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"]
    
    for _ in date:
        races[_] = pd.to_datetime(races[_])
        
    time = ["time","fp1_time", "fp2_time", "fp3_time", "quali_time", "sprint_time"]
    def time_to_decimal(time_str):
        if pd.isna(time_str):
            return np.nan
        h, m, s = map(int, time_str.split(':'))
        return h + m / 60 + s / 3600
    for _ in time:
        races[_]=races[_].apply(time_to_decimal)
    
    loca.rename(columns = {"Grand Prix": "name"}, inplace = True)
    races = pd.merge(races, loca,how = 'left')
    Long = races.Longitude.unique()
    
    avg = []
    for l in Long:
        a = races[races["Longitude"] == l]
        avg_time_A = [a[_].mean() for _ in time]
        avg.append(avg_time_A)
    for j in range(len(avg[0])):
        data = [avg[i][j] for i in range(len(avg))]
        norm = mcolors.Normalize(vmin=min(data), vmax=max(data))  # Normalisation des valeurs
        cmap = plt.cm.plasma
        shapefile_path = "naturalearth_lowres"
        gdf = gpd.read_file(shapefile_path)
        fig, ax = plt.subplots(figsize=(10, 8))
        gdf.plot(ax=ax)
        scatter = ax.scatter(
            races['Longitude'].unique(),
            races['Latitude'].unique(),
            c=data,
            cmap=cmap,  
            s=50,        
            marker='o',  
            edgecolor='k'  
        )
        fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap),
                     ax=ax,
                     orientation='horizontal',
                     label='Moyenne des temps')
        fig.suptitle(f"Horaire moyen de début de {time[j]}")
        plt.show()

In [4]:
def min_max_pit_stop_drivers(nom_pilote: str, pit_stops: pd.DataFrame, drivers : pd.DataFrame, param: {"min", "max"}):
    pit_stops = pit_stops.copy(deep = True)
    pit_stops["duration"] = pit_stops["milliseconds"]*(10**(-3))
    pit_stop_min = pit_stops.groupby(["raceId","driverId"]).agg({"duration": "min"}).reset_index()
    pit_stop_max = pit_stops.groupby(["raceId","driverId"]).agg({"duration": "max"}).reset_index()
    a = pd.merge(drivers, pit_stop_min, on="driverId", how="left")
    a.rename(columns={"duration": "min_pit_stop"}, inplace=True)
    a = pd.merge(a, pit_stop_max, on=["driverId", "raceId"], how="left")
    a.rename(columns={"duration": "max_pit_stop"}, inplace=True)
    a["min_pit_stop"] = a["min_pit_stop"].astype(float)
    a["max_pit_stop"] = a["max_pit_stop"].astype(float)
    a = a[ ~a["min_pit_stop"].isna()]
    a = a[ ~a["max_pit_stop"].isna()]
    drivers_mean_min_pit_stop = a.groupby("driverRef").agg({"min_pit_stop": "mean"}).reset_index()
    drivers_mean_max_pit_stop = a.groupby("driverRef").agg({"max_pit_stop": "mean"}).reset_index()
    if param == 'min':
        return drivers_mean_min_pit_stop[drivers_mean_max_pit_stop["driverRef"] == nom_pilote]
    else:
        return drivers_mean_max_pit_stop[drivers_mean_max_pit_stop["driverRef"] == nom_pilote] 
    

In [5]:
def generer_table_fichier(nom_fichier_recherche):
    import os
    dossier = os.path.join(os.path.expanduser("~"), "Desktop", "donnees_formule_un")
    if not os.path.isdir(dossier):
        raise ValueError("Le fichier 'donnees_formule_un' est introuvable sur votre bureau.")
        print("Le dossier n'existe pas :", dossier)
        return
    for nom_fichier in os.listdir(dossier):
        chemin_fichier = os.path.join(dossier, nom_fichier)
        if os.path.isfile(chemin_fichier):
            nom_sans_ext = os.path.splitext(nom_fichier)[0]
            if nom_sans_ext == nom_fichier_recherche:
                with open(chemin_fichier, 'r', encoding='utf-8') as f:
                    for ligne in f:
                        yield ligne.strip()



In [11]:
driver_standings

,driverStandingsId,raceId,driverId,points,position,positionText,wins
0,1,18,1,10.0,1,1,1
1,2,18,2,8.0,2,2,0
2,3,18,3,6.0,3,3,0
3,4,18,4,5.0,4,4,0
4,5,18,5,4.0,5,5,0
...,...,...,...,...,...,...,...
34590,72867,1132,839,3.0,18,18,0
34591,72868,1132,842,6.0,15,15,0
34592,72869,1132,822,0.0,21,21,0
34593,72870,1132,858,0.0,20,20,0


NameError: name 'seasons' is not defined

In [20]:
count = 1
joueurs_points = dict()
for ligne in generer_table_fichier("driver_standings"):
    begin = 0
    i = 0
    while i < len(ligne):
        if ligne[i] == ',':
            begin = i
            break
        i+=1
    end = 0
    while i < len(ligne):
        if ligne[i] == ",":
            end = i
            break 
        i+=1
    keys = ligne[begin + 1:end]
    victory = ligne[-1]
    if keys in joueurs_points:
        joueurs_points[keys] = joueurs_points[keys] + int(victory)
    else:
        joueurs_point[keys] = int(victory)
    count += 1

ValueError: invalid literal for int() with base 10: 's'

In [15]:

while i < len(a):
    if a[i] == ',':
        end = i 
        break 
    i+=1

In [16]:
end

3